# 🏥 Hospital Multi-Agent Assistant

## Problem Statement
Hospital staff need to query two very different kinds of knowledge in plain English:
- **Structured data** — patient records, billing, admissions (lives in a database)
- **Unstructured knowledge** — hospital policies (lives in documents)

This notebook builds a small multi-agent system with three pieces:

1. **Orchestrator Agent** — classifies each incoming query and routes it to the right specialist
2. **NLP-to-SQL Agent** — turns a question into a `SELECT` query and runs it against a synthetic patient database
3. **RAG Agent** — retrieves relevant chunks from synthetic hospital policy documents and answers from them

### A note on the data sources
The two source links for this project (the Kaggle healthcare dataset and a Drive file with policy
documents) were either not directly reachable from this environment or not shared publicly, so the
patient records and policy documents below are **synthetically generated to match the same schema/style**
described in those resources. The Kaggle dataset's real column layout (Name, Age, Gender, Blood Type,
Medical Condition, Date of Admission, Doctor, Hospital, Insurance Provider, Billing Amount, Room Number,
Admission Type, Discharge Date, Medication, Test Results) is what the `patients` table below follows, so
swapping in the real CSV later is a one-line change (see the last section).

### Design choice: local models with graceful fallback
Per the brief, this uses **local/free models** rather than a paid API:
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- Generation: `google/flan-t5-base`

Both are downloaded from Hugging Face the first time you run this in an environment with internet
access (e.g. Colab). If they're not available (no internet, first run, etc.), every agent below
**automatically falls back** to a lightweight local-only method (TF-IDF similarity for retrieval/routing,
and a rule-based parser for SQL generation) so the notebook still runs end-to-end anywhere.

## ⚙️ Setup

In [1]:
!pip -q install faker sentence-transformers transformers torch scikit-learn 2>/dev/null

import os, re, glob, sqlite3, random
import numpy as np
import pandas as pd


## Part 1 — Synthetic Patient Database

Generates patient records matching the Kaggle healthcare dataset's schema and loads them into a
local SQLite database (`hospital.db`, table `patients`).

In [2]:
from faker import Faker

fake = Faker()
random.seed(42)
Faker.seed(42)

GENDERS = ["Male", "Female"]
BLOOD_TYPES = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
CONDITIONS = ["Diabetes", "Hypertension", "Asthma", "Obesity", "Arthritis", "Cancer"]
HOSPITALS = ["St. Mary's Hospital", "Green Valley Medical Center", "Lakeside General Hospital",
             "Riverside Health Clinic", "Sunrise Community Hospital"]
INSURANCE = ["Aetna", "Blue Cross", "Cigna", "UnitedHealthcare", "Medicare"]
ADMISSION_TYPES = ["Emergency", "Elective", "Urgent"]
MEDICATIONS = ["Aspirin", "Ibuprofen", "Paracetamol", "Metformin", "Lipitor", "Penicillin"]
TEST_RESULTS = ["Normal", "Abnormal", "Inconclusive"]

def generate_patients(n=300):
    rows = []
    for i in range(1, n + 1):
        admission_date = fake.date_between(start_date="-2y", end_date="today")
        discharge_date = fake.date_between(start_date=admission_date, end_date="today")
        rows.append({
            "patient_id": i,
            "name": fake.name(),
            "age": random.randint(1, 95),
            "gender": random.choice(GENDERS),
            "blood_type": random.choice(BLOOD_TYPES),
            "medical_condition": random.choice(CONDITIONS),
            "date_of_admission": admission_date.isoformat(),
            "doctor": "Dr. " + fake.last_name(),
            "hospital": random.choice(HOSPITALS),
            "insurance_provider": random.choice(INSURANCE),
            "billing_amount": round(random.uniform(500, 50000), 2),
            "room_number": random.randint(100, 599),
            "admission_type": random.choice(ADMISSION_TYPES),
            "discharge_date": discharge_date.isoformat(),
            "medication": random.choice(MEDICATIONS),
            "test_results": random.choice(TEST_RESULTS),
        })
    return pd.DataFrame(rows)

patients_df = generate_patients(300)

conn = sqlite3.connect("hospital.db")
patients_df.to_sql("patients", conn, if_exists="replace", index=False)
conn.commit()

print(patients_df.shape, "rows loaded into hospital.db -> patients")
patients_df.head()


(300, 16) rows loaded into hospital.db -> patients


,patient_id,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,1,Daniel Doyle,82,Male,A+,Cancer,2025-10-21,Dr. Rhodes,Lakeside General Hospital,Blue Cross,11548.93,477,Emergency,2025-10-27,Penicillin,Inconclusive
1,2,Allen Robinson,70,Male,O+,Diabetes,2026-04-24,Dr. Wagner,St. Mary's Hospital,Aetna,11322.58,358,Urgent,2026-04-30,Aspirin,Inconclusive
2,3,Kimberly Robinson,26,Female,B-,Obesity,2024-07-30,Dr. Lawrence,Sunrise Community Hospital,Cigna,40566.81,103,Emergency,2024-12-18,Penicillin,Abnormal
3,4,Melissa Peterson,44,Female,B+,Hypertension,2026-02-22,Dr. Moore,Lakeside General Hospital,Aetna,5090.92,149,Elective,2026-02-22,Paracetamol,Inconclusive
4,5,Brent Abbott,34,Male,O-,Arthritis,2026-06-10,Dr. Munoz,St. Mary's Hospital,UnitedHealthcare,4400.61,250,Urgent,2026-06-20,Lipitor,Abnormal


## Part 2 — Synthetic Hospital Policy Documents

A handful of policy documents covering the topics staff actually get asked about: visiting hours,
patient privacy, discharge, billing/insurance, infection control, and emergency admission.

**To use real policy documents instead:** drop `.txt` files into the `policies/` folder (replacing
or adding to these) and re-run from Part 3 onward — nothing else needs to change.

In [3]:
os.makedirs("policies", exist_ok=True)

POLICIES = {
"visiting_hours_policy.txt": """Visiting Hours Policy

General visiting hours across all departments are 10:00 AM to 8:00 PM daily. The Intensive Care Unit (ICU) has restricted visiting hours of 11:00 AM to 1:00 PM and 5:00 PM to 7:00 PM, limited to two visitors per patient at a time. The Pediatric ward allows parents or legal guardians to stay overnight, but other visitors must leave by 8:00 PM.

Maternity ward visiting hours are 9:00 AM to 9:00 PM, with only one support person permitted to stay overnight with the mother. Visitors showing signs of fever, cough, or any contagious illness will not be permitted entry, in line with infection control guidelines.

All visitors must sign in at the reception desk and wear a visitor badge at all times while on hospital premises. Children under 12 are only allowed to visit immediate family members and must be accompanied by an adult. During flu season (typically November through March), the hospital may further restrict visiting hours or the number of visitors per patient; any such changes will be posted at the main entrance and on the hospital website.

Visitors are asked to keep noise to a minimum, silence mobile phones, and avoid bringing large groups into shared patient rooms.""",

"patient_privacy_policy.txt": """Patient Data Privacy Policy

The hospital is committed to protecting the confidentiality of all patient information in accordance with applicable healthcare privacy regulations. Patient medical records, including diagnosis, treatment history, billing information, and test results, are considered confidential and may only be accessed by authorized medical staff directly involved in a patient's care.

Patients or their legal representatives may request a copy of their medical records by submitting a written request to the Health Information Management (HIM) department. Requests are typically processed within 5 to 10 business days. A valid photo ID is required to release records, and a small administrative fee may apply for physical copies exceeding 20 pages.

Patient information will not be disclosed to third parties, including family members, without the patient's written consent, except in cases required by law (such as public health reporting) or medical emergencies where disclosure is necessary for treatment. Patients have the right to request corrections to inaccurate information in their records and to receive a list of disclosures made from their records in the preceding six years.

Any suspected breach of patient data must be reported immediately to the Privacy Officer, who will investigate and notify affected patients as required by law.""",

"discharge_policy.txt": """Patient Discharge Policy

Discharge planning begins at the time of admission and is coordinated by the attending physician, nursing staff, and case management team. A patient is considered medically ready for discharge once the physician determines that further inpatient care is not required.

On the day of discharge, patients will receive a discharge summary detailing their diagnosis, treatment received, medications prescribed, and any follow-up appointments or instructions. Discharge typically occurs before 11:00 AM to allow adequate time for room turnover; patients requiring additional time must coordinate with the nursing station in advance.

Patients who wish to leave the hospital against medical advice (AMA) must sign an AMA form acknowledging the risks explained by their physician. For patients requiring continued care after leaving, the case management team will arrange transfer to a rehabilitation facility, skilled nursing facility, or home healthcare services as appropriate.

All outstanding billing matters should ideally be settled before discharge, though this is not required to receive necessary medical care or to be discharged. Patients are encouraged to schedule a follow-up appointment with their primary care physician within 7 to 14 days of discharge.""",

"billing_insurance_policy.txt": """Billing and Insurance Policy

The hospital accepts a range of insurance providers including Aetna, Blue Cross, Cigna, UnitedHealthcare, and Medicare. Patients are responsible for verifying that their insurance plan covers the services they will receive prior to elective procedures; emergency care will never be delayed pending insurance verification.

An itemized bill will be provided to every patient upon request, detailing charges for room and board, procedures, medications, and any additional services rendered during their stay. Patients who believe there is an error on their bill may file a billing dispute with the Patient Financial Services department within 60 days of the statement date.

For patients without insurance or with limited coverage, the hospital offers a financial assistance program based on income and household size; applications are available through the billing office or hospital website. Payment plans can be arranged for outstanding balances, typically spread over 6 to 24 months depending on the amount owed.

Co-payments and deductibles are generally collected at the time of service for scheduled visits. Failure to pay outstanding balances may result in the account being referred to a collections agency after 120 days of non-payment, following at least three written notices.""",

"infection_control_policy.txt": """Infection Control and Hygiene Policy

All staff, patients, and visitors are required to perform hand hygiene using soap and water or alcohol-based sanitizer before and after any patient contact. Personal protective equipment (PPE), including gloves, gowns, and masks, must be worn by staff when treating patients on isolation precautions.

Patients diagnosed with airborne, droplet, or contact-transmissible infections will be placed in isolation rooms, and visitor access will be restricted per the isolation category assigned by infection control staff. Any staff member exhibiting symptoms of a contagious illness must not report to work and should notify their supervisor and Employee Health immediately.

The hospital conducts routine environmental cleaning of all patient rooms, with terminal cleaning performed after each patient discharge, particularly for rooms previously occupied by patients on isolation precautions. Surgical and procedural areas follow stricter sterilization protocols in accordance with hospital infection prevention guidelines.

Patients and visitors are encouraged to report any concerns about cleanliness or hygiene practices to the nursing staff or the Infection Prevention department directly.""",

"emergency_admission_policy.txt": """Emergency Admission Policy

Patients arriving through the Emergency Department are triaged upon arrival based on the severity of their condition, using a standardized 5-level triage scale, with Level 1 representing immediate life-threatening conditions and Level 5 representing non-urgent cases. Triage assessment typically occurs within 10 minutes of arrival.

Emergency admissions do not require prior insurance authorization; care will be provided regardless of a patient's ability to pay or insurance status, in compliance with emergency medical treatment regulations. Once stabilized, patients may be admitted as inpatients if further treatment is required, transferred to another facility if a higher level of care is needed, or discharged with follow-up instructions.

Family members will be updated on a patient's condition as soon as possible, subject to the patient's privacy preferences. Patients admitted through the Emergency Department will be assigned an inpatient room based on bed availability and medical needs, which may involve a temporary stay in an observation unit.

The Emergency Department operates 24 hours a day, 7 days a week, and is staffed with emergency physicians, nurses, and support personnel at all times.""",
}

for filename, content in POLICIES.items():
    with open(os.path.join("policies", filename), "w") as f:
        f.write(content)

print(f"Wrote {len(POLICIES)} policy documents to policies/")
for fname in sorted(os.listdir("policies")):
    print(" -", fname)


Wrote 6 policy documents to policies/
 - billing_insurance_policy.txt
 - discharge_policy.txt
 - emergency_admission_policy.txt
 - infection_control_policy.txt
 - patient_privacy_policy.txt
 - visiting_hours_policy.txt


## Part 3 — Chunking + Embeddings

Policy documents get split into overlapping word chunks, then embedded so we can retrieve the most
relevant ones for a given question. The `Embedder` class tries `sentence-transformers` first and
transparently drops to TF-IDF if that's not available — either way, the rest of the pipeline is
identical.

In [4]:
def load_policy_chunks(folder="policies", chunk_size=110, overlap=25):
    chunks = []
    for path in sorted(glob.glob(f"{folder}/*.txt")):
        doc_name = os.path.basename(path)
        text = open(path).read()
        words = text.split()
        start = 0
        while start < len(words):
            piece = " ".join(words[start:start + chunk_size])
            chunks.append({"doc": doc_name, "text": piece})
            if start + chunk_size >= len(words):
                break
            start += chunk_size - overlap
    return chunks

policy_chunks = load_policy_chunks()
chunk_texts = [c["text"] for c in policy_chunks]
print(f"{len(policy_chunks)} policy chunks loaded")


14 policy chunks loaded


In [5]:
class Embedder:
    """Wraps sentence-transformers if it can be loaded; otherwise falls back
    to a TF-IDF vectorizer fit on the corpus it's given. Same .encode() interface
    either way, so nothing downstream needs to know which backend is active."""

    def __init__(self, fallback_corpus):
        self.backend = None
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.backend = "sentence-transformers"
        except Exception as e:
            print(f"[embedder] sentence-transformers unavailable ({type(e).__name__}). Falling back to TF-IDF.")
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.vectorizer.fit(fallback_corpus)
            self.backend = "tfidf"

    def encode(self, texts):
        if self.backend == "sentence-transformers":
            return np.array(self.model.encode(texts, normalize_embeddings=True))
        vecs = self.vectorizer.transform(texts).toarray()
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return vecs / norms


# seed queries used both for retrieval-quality bootstrapping and (later) orchestrator routing
ROUTING_SEED_SQL = [
    "How many patients have diabetes?",
    "What is the average billing amount?",
    "List patients admitted in the last month",
    "Which doctor treats the most patients?",
    "Show me patients with abnormal test results",
    "How many patients are covered by Medicare?",
]
ROUTING_SEED_RAG = [
    "What are the visiting hours?",
    "What is the hospital's policy on patient data privacy?",
    "How do I request a copy of my medical records?",
    "What is the discharge procedure?",
    "What insurance documentation is required for billing?",
    "What are the infection control rules for visitors?",
]

embedder = Embedder(fallback_corpus=chunk_texts + ROUTING_SEED_SQL + ROUTING_SEED_RAG)
chunk_embeddings = embedder.encode(chunk_texts)
print("embedding backend in use:", embedder.backend)


[embedder] sentence-transformers unavailable (ModuleNotFoundError). Falling back to TF-IDF.


embedding backend in use: tfidf


In [6]:
def retrieve_chunks(query, k=3):
    q_emb = embedder.encode([query])[0]
    scores = chunk_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:k]
    return [(policy_chunks[i], float(scores[i])) for i in top_idx]

# sanity check
for chunk, score in retrieve_chunks("What time can I visit a patient in the ICU?", k=3):
    print(round(score, 3), chunk["doc"])


0.16 discharge_policy.txt
0.1 visiting_hours_policy.txt
0.085 visiting_hours_policy.txt


## Part 4 — RAG Agent

Retrieves the top-k policy chunks for a query, then either hands them to `flan-t5-base` to write a
grounded answer, or — if no local LLM is available — falls back to pulling the most relevant
sentences straight out of the top chunk (`extractive_answer`).

In [7]:
class Generator:
    """Wraps a local text2text LLM if one can be downloaded; otherwise
    backend == 'rule-based' and callers should use their own fallback logic."""

    def __init__(self):
        try:
            from transformers import pipeline
            self.pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=200)
            self.backend = "flan-t5-base"
        except Exception as e:
            print(f"[generator] flan-t5-base unavailable ({type(e).__name__}). Falling back to rule-based generation.")
            self.backend = "rule-based"

    def generate(self, prompt: str) -> str:
        if self.backend == "flan-t5-base":
            return self.pipe(prompt)[0]["generated_text"].strip()
        raise RuntimeError("No LLM backend loaded; use the fallback path instead.")


def extractive_answer(query, retrieved, max_sentences=2):
    """Fallback answer synthesis: pull the sentences most relevant to the
    query out of the top retrieved chunk, instead of generating free text."""
    if not retrieved:
        return "I couldn't find anything relevant to that in the policy documents."
    top_chunk_text = retrieved[0][0]["text"]
    sentences = re.split(r"(?<=[.!?])\s+", top_chunk_text)
    q_words = set(w.lower() for w in re.findall(r"\w+", query) if len(w) > 3)
    scored = []
    for s in sentences:
        s_words = set(w.lower() for w in re.findall(r"\w+", s))
        scored.append((len(q_words & s_words), s))
    scored.sort(key=lambda x: x[0], reverse=True)
    best = [s for _, s in scored[:max_sentences] if s.strip()]
    return " ".join(best or sentences[:max_sentences]).strip()


def rag_agent(query, generator=None, k=3):
    retrieved = retrieve_chunks(query, k=k)
    sources = sorted(set(c["doc"] for c, _ in retrieved))

    if generator is not None and generator.backend != "rule-based":
        context = "\n\n".join(c["text"] for c, _ in retrieved)
        prompt = (
            "Answer the question using ONLY the context below. "
            "If the answer isn't in the context, say you don't have that information.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        )
        answer, answer_source = generator.generate(prompt), "llm"
    else:
        answer, answer_source = extractive_answer(query, retrieved), "extractive"

    return {"answer": answer, "answer_source": answer_source, "sources": sources}


## Part 5 — NLP-to-SQL Agent

Turns a question into a `SELECT` query against `patients`. There are two layers of safety here:
1. A schema + few-shot prompt constrains the LLM to `SELECT`-only queries in the first place
2. `is_safe_sql()` double-checks the generated query before it's ever executed — anything that isn't a
   single `SELECT`, or that contains a write/DDL keyword, is refused

If no local LLM is loaded, `rule_based_nl_to_sql` handles the common question shapes (counts,
averages, filters by condition/doctor/insurer/etc.) with plain pattern matching.

In [8]:
SCHEMA_DESC = """Table: patients
Columns:
- patient_id (INTEGER)
- name (TEXT)
- age (INTEGER)
- gender (TEXT: 'Male' or 'Female')
- blood_type (TEXT)
- medical_condition (TEXT: e.g. 'Diabetes', 'Hypertension', 'Asthma', 'Obesity', 'Arthritis', 'Cancer')
- date_of_admission (TEXT, 'YYYY-MM-DD')
- doctor (TEXT)
- hospital (TEXT)
- insurance_provider (TEXT: 'Aetna', 'Blue Cross', 'Cigna', 'UnitedHealthcare', 'Medicare')
- billing_amount (REAL)
- room_number (INTEGER)
- admission_type (TEXT: 'Emergency', 'Elective', 'Urgent')
- discharge_date (TEXT, 'YYYY-MM-DD')
- medication (TEXT)
- test_results (TEXT: 'Normal', 'Abnormal', 'Inconclusive')
"""

CONDITIONS_L = [c.lower() for c in CONDITIONS]
ADMISSION_TYPES_L = [a.lower() for a in ADMISSION_TYPES]
GENDERS_L = [g.lower() for g in GENDERS]
BLOCKED_KEYWORDS = ["insert", "update", "delete", "drop", "alter", "attach", "pragma", "create", "--", ";--"]


def is_safe_sql(sql: str) -> bool:
    lowered = sql.strip().lower()
    if not lowered.startswith("select"):
        return False
    if any(bad in lowered for bad in BLOCKED_KEYWORDS):
        return False
    if sql.count(";") > 1:
        return False
    return True


def rule_based_nl_to_sql(question: str) -> str:
    q = question.lower()

    def has_word(word, text=q):
        return re.search(rf"\b{re.escape(word)}\b", text) is not None

    if "how many" in q or "count" in q or "number of" in q:
        select_clause = "SELECT COUNT(*) AS count"
    elif "average" in q or "avg" in q:
        select_clause = "SELECT AVG(age) AS average_age" if has_word("age") else "SELECT AVG(billing_amount) AS average_billing"
    elif "total" in q and "billing" in q:
        select_clause = "SELECT SUM(billing_amount) AS total_billing"
    elif "list" in q or "who" in q or "show" in q or "which patients" in q:
        select_clause = "SELECT name, medical_condition, doctor, hospital"
    else:
        select_clause = "SELECT *"

    filters = []
    for cond in CONDITIONS_L:
        if has_word(cond):
            filters.append(f"medical_condition = '{cond.capitalize()}'")
            break
    for adm in ADMISSION_TYPES_L:
        if has_word(adm):
            filters.append(f"admission_type = '{adm.capitalize()}'")
            break
    # check 'abnormal' before 'normal' -- 'normal' is a substring of 'abnormal'
    if has_word("abnormal"):
        filters.append("test_results = 'Abnormal'")
    elif has_word("inconclusive"):
        filters.append("test_results = 'Inconclusive'")
    elif has_word("normal"):
        filters.append("test_results = 'Normal'")
    for g in GENDERS_L:
        if has_word(g):
            filters.append(f"gender = '{g.capitalize()}'")
            break
    for provider in INSURANCE:
        if provider.lower() in q:
            filters.append(f"insurance_provider = '{provider}'")
            break

    doctor_match = re.search(r"dr\.?\s+([a-z]+)", q)
    if doctor_match:
        filters.append(f"doctor LIKE '%{doctor_match.group(1).capitalize()}%'")

    sql = f"{select_clause} FROM patients"
    if filters:
        sql += " WHERE " + " AND ".join(filters)
    if select_clause.startswith("SELECT name"):
        sql += " LIMIT 20"
    return sql + ";"


def sql_agent(question: str, generator=None):
    if generator is not None and generator.backend != "rule-based":
        prompt = (
            "Convert the question into a single SQLite SELECT query.\n"
            f"{SCHEMA_DESC}\nOnly use SELECT statements, never modify data.\n\n"
            f"Q: {question}\nSQL:"
        )
        raw_sql = generator.generate(prompt)
        sql = (raw_sql.split(";")[0].strip() + ";") if raw_sql else rule_based_nl_to_sql(question)
        sql_source = "llm"
    else:
        sql = rule_based_nl_to_sql(question)
        sql_source = "rule-based"

    if not is_safe_sql(sql):
        return {"sql": sql, "sql_source": sql_source, "error": "Query failed the safety check, refused to run.", "result": None}

    try:
        result_df = pd.read_sql_query(sql, conn)
        return {"sql": sql, "sql_source": sql_source, "error": None, "result": result_df}
    except Exception as e:
        return {"sql": sql, "sql_source": sql_source, "error": str(e), "result": None}


## Part 6 — Orchestrator Agent

Classifies a query by comparing its embedding to two small sets of example queries (one SQL-style,
one policy-style) and routes to whichever set it's closer to. This reuses the same `Embedder`
instance from Part 3, so it inherits the same fallback behavior automatically.

In [9]:
sql_seed_emb = embedder.encode(ROUTING_SEED_SQL)
rag_seed_emb = embedder.encode(ROUTING_SEED_RAG)

def classify_query(query):
    q_emb = embedder.encode([query])[0]
    sql_score = float((sql_seed_emb @ q_emb).max())
    rag_score = float((rag_seed_emb @ q_emb).max())
    route = "sql" if sql_score >= rag_score else "rag"
    return route, sql_score, rag_score


generator = Generator()  # shared by both agents; falls back automatically if offline

def orchestrator(query: str) -> dict:
    route, sql_score, rag_score = classify_query(query)
    if route == "sql":
        result = sql_agent(query, generator=generator)
        return {"agent": "NLP-to-SQL Agent", "query": query, "route_confidence": round(sql_score, 3), **result}
    else:
        result = rag_agent(query, generator=generator)
        return {"agent": "RAG Agent", "query": query, "route_confidence": round(rag_score, 3), **result}


[generator] flan-t5-base unavailable (OSError). Falling back to rule-based generation.


## Part 7 — Demo

A mix of data questions and policy questions, run through the single `orchestrator()` entry point.

In [10]:
demo_queries = [
    "How many patients have diabetes?",
    "What are the hospital's visiting hours?",
    "What is the average billing amount for emergency admissions?",
    "How do I request a copy of my medical records?",
    "What is the hospital's discharge policy?",
    "How many patients are covered by Medicare?",
    "What are the infection control rules for visitors?",
]

for q in demo_queries:
    out = orchestrator(q)
    print(f"QUERY: {q}")
    print(f"  -> routed to: {out['agent']}  (confidence {out['route_confidence']})")
    if out["agent"] == "NLP-to-SQL Agent":
        print(f"  SQL ({out['sql_source']}): {out['sql']}")
        if out["error"]:
            print(f"  error: {out['error']}")
        else:
            print(out["result"].to_string(index=False))
    else:
        print(f"  answer ({out['answer_source']}): {out['answer']}")
        print(f"  sources: {out['sources']}")
    print("-" * 70)


QUERY: How many patients have diabetes?
  -> routed to: NLP-to-SQL Agent  (confidence 1.0)
  SQL (rule-based): SELECT COUNT(*) AS count FROM patients WHERE medical_condition = 'Diabetes';
 count
    48
----------------------------------------------------------------------
QUERY: What are the hospital's visiting hours?
  -> routed to: RAG Agent  (confidence 0.902)
  answer (extractive): Visiting Hours Policy General visiting hours across all departments are 10:00 AM to 8:00 PM daily. The Intensive Care Unit (ICU) has restricted visiting hours of 11:00 AM to 1:00 PM and 5:00 PM to 7:00 PM, limited to two visitors per patient at a time.
  sources: ['emergency_admission_policy.txt', 'visiting_hours_policy.txt']
----------------------------------------------------------------------
QUERY: What is the average billing amount for emergency admissions?
  -> routed to: NLP-to-SQL Agent  (confidence 0.681)
  SQL (rule-based): SELECT AVG(billing_amount) AS average_billing FROM patients WHERE admis

## 🚀 Bonus: Interactive Mode

Same pattern as the single-agent assignment — a simple loop for manual testing. Not auto-run here
since it blocks on `input()`; run this cell yourself to try it live.

In [ ]:
while True:
    user_input = input("Ask about patients or policy (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    out = orchestrator(user_input)
    print(f"[{out['agent']}]")
    if out["agent"] == "NLP-to-SQL Agent":
        print(out["sql"])
        print(out["error"] or out["result"])
    else:
        print(out["answer"])
        print("Sources:", out["sources"])


## Swapping in the real data

- **Real patient data**: replace the `generate_patients()` call with `pd.read_csv("healthcare_dataset.csv")`
  (download from the [Kaggle dataset](https://www.kaggle.com/datasets/prasad22/healthcare-dataset)),
  lowercase/rename columns to match the schema above, then re-run the `to_sql(...)` cell.
- **Real policy documents**: drop `.txt` (or `.pdf`, with a small extra extraction step) files into
  `policies/`, re-run from Part 3 onward.
- **Better generation quality**: run this notebook somewhere with internet access to Hugging Face
  (e.g. Google Colab) so `Embedder`/`Generator` load the real models instead of the fallbacks — retrieval
  and answer quality both improve noticeably over TF-IDF/extractive matching.